In [2]:
# Azure Machine Learning workspace details:
subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
resource_group = 'msan-aml'

workspace = 'msan-retrieval-ranking-aml'

datastore_name = 'bingads_algo_prod_networkprotection_c08'
path = 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1Ranking/Data/DeepFBS_V3_20250416_20250422_20X/training_prepare/EventEmb.tsv'

# long-form Datastore uri format:
uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path}'

import pandas as pd

df = pd.read_csv(uri, sep='\t', nrows=10, header=None, names=['event_id', 'event_title', 'event_embedding'])


Timeout was exceeded in force_flush().
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Resolving access token for scope "https://datalake.azure.net//.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration


In [3]:
df

,event_id,event_title,event_embedding
0,1,EdgePageTitle,-0.14902773 -0.31867605 -0.48099414 -0.1400186...
1,2,SearchClick,-0.19081563 -0.12149302 0.122159675 0.17811456...
2,3,EdgePageTitle,-0.33395073 -0.12832756 0.41955265 0.23110661 ...
3,4,EdgePageTitle,-0.32775766 -0.43254247 0.025795273 0.17582624...
4,5,SearchClick,0.18657781 0.02512318 0.32489547 0.14523757 -0...
5,6,EdgePageTitle,-0.19428764 -0.06516099 -0.13053034 -0.0724147...
6,7,EdgePageTitle,0.19964632 -0.03098667 0.18544452 -0.057399716...
7,8,EdgeSearchQuery,-0.0746341 0.10937274 -0.0286799 0.19602679 -0...
8,9,EdgePageTitle,0.20108794 -0.2186131 -0.500851 -0.007621092 0...
9,10,EdgePageTitle,0.23336093 -0.3679636 -0.18904611 0.08037635 0...


In [5]:
import numpy as np

embeddings = df['event_embedding'].apply(lambda x: np.fromstring(x, sep=' ', dtype=np.float32))
embeddings_array = np.vstack(embeddings.values)

In [9]:
embeddings_array.shape

(10, 64)

In [8]:
from azureml.fsspec import AzureMachineLearningFileSystem
import tempfile
import shutil

subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
resource_group = 'msan-aml'
workspace = 'msan-retrieval-ranking-aml'
datastore_name = 'bingads_algo_prod_networkprotection_c08'
uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}'
fs = AzureMachineLearningFileSystem(uri)

path = '/shares/bingads.hm/local/NativeAds/Relevance/Data/sequential/hstu/v2/musing_orange_133g6bjvjd/ads/shard_0.npy'

# Create a temporary file to store the downloaded .npy
with fs.open(path, "rb") as file:

    import numpy as np
    array = np.load(file)

KeyboardInterrupt: 

In [16]:
# read the first row of the array
array[:21]

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.22213192,  0.15244299,  0.1104566 , ..., -0.1839646 ,
         0.09274498,  0.05349246]], dtype=float32)

In [ ]:
# Demonstrate the difference between two approaches

# Method 1: Direct processing (requires .values)
embeddings_direct = df['event_embedding'].apply(lambda x: np.fromstring(x, sep=' ', dtype=np.float32))
print("Method 1 - Direct processing:")
print(f"embeddings_direct type: {type(embeddings_direct)}")
print(f"Requires .values: {type(embeddings_direct.values)}")

# Method 2: Chunk-based approach (doesn't require .values)
all_embeddings = []
for i in range(len(df)):
    chunk_embeddings = df.iloc[i:i+1]['event_embedding'].apply(lambda x: np.fromstring(x, sep=' ', dtype=np.float32))
    all_embeddings.extend(chunk_embeddings)  # extend unpacks the items

print(f"\nMethod 2 - Chunk-based approach:")
print(f"all_embeddings type: {type(all_embeddings)}")
print(f"First element type: {type(all_embeddings[0])}")

# Verify that both methods produce the same result
result1 = np.vstack(embeddings_direct.values)
result2 = np.vstack(all_embeddings)
print(f"\nResults are identical: {np.array_equal(result1, result2)}")
print(f"Shape1: {result1.shape}, Shape2: {result2.shape}")